# TE03 - Image Segmentation via Thresholding

## Reference Links

- OpenCV docs: http://docs.opencv.org/3.2.0/
- OpenCV thresholding tutorial: https://docs.opencv.org/3.2.0/d7/d4d/tutorial_py_thresholding.html
- NumPy docs: http://www.numpy.org/
- SciPy docs: https://docs.scipy.org/doc/
- Matplotlib docs: http://matplotlib.org/


## Setup

Expected files:
- `../data/te03/voilier_oies_blanches.jpg`
- `../data/te03/img_ds.jpg`


In [ ]:
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = Path('../data/te03')
OUT_DIR = Path('../outputs/te03')
OUT_DIR.mkdir(parents=True, exist_ok=True)

required_files = ['voilier_oies_blanches.jpg', 'img_ds.jpg']
missing = [f for f in required_files if not (DATA_DIR / f).exists()]
if missing:
    raise FileNotFoundError(f'Missing files: {missing}')


def show_gray(image, title):
    plt.figure(figsize=(6, 4))
    plt.imshow(image, cmap='gray')
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()


def compute_histogram(image):
    return cv2.calcHist([image], [0], None, [256], [0, 256]).ravel()


def auto_foreground_otsu(image):
    threshold, binary = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if (binary > 0).mean() > 0.5:
        binary = cv2.bitwise_not(binary)
    return threshold, binary


def run_camera_session(out_dir, window_name, *, allow_snapshot=False, record_path='', transform=lambda frame: frame):
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print('Webcam unavailable on index 0.')
        return

    snapshot_idx = 1
    writer = cv2.VideoWriter()

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                print('Camera stream ended.')
                break

            display_frame = transform(frame)
            if record_path and not writer.isOpened():
                height, width = display_frame.shape[:2]
                is_color = display_frame.ndim == 3
                fourcc = cv2.VideoWriter_fourcc(*'XVID')
                writer.open(str(record_path), fourcc, 20.0, (width, height), isColor=is_color)

            cv2.imshow(window_name, display_frame)
            if writer.isOpened():
                writer.write(display_frame)

            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):
                break
            if allow_snapshot and key == ord('s'):
                snapshot_path = out_dir / f'fra{snapshot_idx}.png'
                cv2.imwrite(str(snapshot_path), display_frame)
                print(f'Saved {snapshot_path.name}')
                snapshot_idx += 1
    finally:
        cap.release()
        if writer.isOpened():
            writer.release()
        cv2.destroyAllWindows()


print('Setup OK')


## 1. Segmentation and Binarization Theory (Summary)

Key points from statement:
- Binarization segments an image into two classes.
- Binary image has only two gray levels (class labels).
- Thresholding rule with threshold `T`:

`J(x, y) = 1 if I(x, y) > T, else 0`

- If objects are darker than background, invert comparison sign.
- `T` may be manual (heuristic) or automatic (e.g., Otsu), global or local.
- Binarization is followed by connected-component extraction / region labeling.


### Theory answers

- Segmentation is fundamental because it turns raw intensities into regions or objects that can be measured, classified, or tracked.
- Global thresholding becomes unreliable when illumination varies, classes overlap in the histogram, or noise creates false bridges between regions.
- Local thresholding is preferable when contrast changes across the image or when the background is not spatially uniform.


## 2. Experimentation: Grayscale Binarization and Trackbars

### 2.1 Manual thresholding

Tasks:
1. Convert `voilier_oies_blanches.jpg` to grayscale (`cv2.cvtColor` or direct grayscale read).
2. Apply manual threshold with `cv2.threshold` (choose `T`).
3. Test and understand different `threshold_type` options.
4. Discuss why threshold-based segmentation may be relevant for this image.
5. Validate your argument using image histogram.


In [ ]:
img_gray = cv2.imread(str(DATA_DIR / 'voilier_oies_blanches.jpg'), cv2.IMREAD_GRAYSCALE)
if img_gray is None:
    raise FileNotFoundError('Unable to read voilier_oies_blanches.jpg')

manual_threshold = 130
threshold_results = {
    'binary': cv2.threshold(img_gray, manual_threshold, 255, cv2.THRESH_BINARY)[1],
    'binary_inv': cv2.threshold(img_gray, manual_threshold, 255, cv2.THRESH_BINARY_INV)[1],
    'trunc': cv2.threshold(img_gray, manual_threshold, 255, cv2.THRESH_TRUNC)[1],
    'tozero': cv2.threshold(img_gray, manual_threshold, 255, cv2.THRESH_TOZERO)[1],
}
img_binary_inv = threshold_results['binary_inv']

plt.figure(figsize=(12, 8))
plt.subplot(2, 3, 1)
plt.imshow(img_gray, cmap='gray')
plt.title('voilier_oies_blanches.jpg')
plt.axis('off')

for idx, (name, image) in enumerate(threshold_results.items(), start=2):
    plt.subplot(2, 3, idx)
    plt.imshow(image, cmap='gray')
    plt.title(f'{name} at T={manual_threshold}')
    plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
hist_voilier = compute_histogram(img_gray)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.imshow(img_gray, cmap='gray')
plt.title('Grayscale voilier image')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.plot(hist_voilier, color='tab:blue')
plt.axvline(manual_threshold, color='tab:red', linestyle='--', label=f'T = {manual_threshold}')
plt.title('Histogram of voilier_oies_blanches.jpg')
plt.xlabel('Gray level')
plt.ylabel('Pixels')
plt.legend()
plt.tight_layout()
plt.show()

print('The birds are darker than the sky, so THRESH_BINARY_INV keeps them as white foreground while the brighter background stays black.')


### 2.2 Manual thresholding with trackbar

Tasks:
1. Attach a trackbar to the thresholding window (`cv2.createTrackbar`).
2. Move thresholding operation into a callback function.
3. Read threshold value with `cv2.getTrackbarPos`.
4. Verify binary output updates as slider changes.


In [ ]:
RUN_TRACKBAR_DEMO = False


def run_threshold_trackbar(image, initial_threshold=130):
    window_name = 'TE03 Threshold Trackbar'

    def update(_):
        threshold_value = cv2.getTrackbarPos('T', window_name)
        binary = cv2.threshold(image, threshold_value, 255, cv2.THRESH_BINARY_INV)[1]
        cv2.imshow(window_name, binary)

    try:
        cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
        cv2.createTrackbar('T', window_name, initial_threshold, 255, update)
        update(initial_threshold)
        while True:
            key = cv2.waitKey(30) & 0xFF
            if key == ord('q'):
                break
    except cv2.error as exc:
        print(f'Trackbar demo skipped: {exc}')
    finally:
        cv2.destroyAllWindows()


if RUN_TRACKBAR_DEMO:
    run_threshold_trackbar(img_gray)
else:
    print("Set RUN_TRACKBAR_DEMO = True to explore the inverse threshold interactively. Press 'q' to quit.")


### 2.3 Interpretation answers

- As `T` increases, more pixels become white with `THRESH_BINARY_INV`, so the bird regions grow and background details start leaking into the mask.
- On `img_ds.jpg`, a single global threshold leaves unstable boundaries because several regions share overlapping gray levels.
- A light Gaussian blur improves the result by reducing local fluctuations before thresholding.


In [ ]:
img_ds_gray = cv2.imread(str(DATA_DIR / 'img_ds.jpg'), cv2.IMREAD_GRAYSCALE)
if img_ds_gray is None:
    raise FileNotFoundError('Unable to read img_ds.jpg')

img_ds_blurred = cv2.GaussianBlur(img_ds_gray, (5, 5), 0)
threshold_img_ds = 174
img_ds_raw = cv2.threshold(img_ds_gray, threshold_img_ds, 255, cv2.THRESH_BINARY)[1]
img_ds_blurred_binary = cv2.threshold(img_ds_blurred, threshold_img_ds, 255, cv2.THRESH_BINARY)[1]

plt.figure(figsize=(12, 8))
comparisons = [
    (img_ds_gray, 'img_ds grayscale'),
    (img_ds_raw, f'Raw threshold at T={threshold_img_ds}'),
    (img_ds_blurred, 'Gaussian blur 5x5'),
    (img_ds_blurred_binary, f'Blur + threshold at T={threshold_img_ds}'),
]
for idx, (image, title) in enumerate(comparisons, start=1):
    plt.subplot(2, 2, idx)
    plt.imshow(image, cmap='gray')
    plt.title(title)
    plt.axis('off')

plt.tight_layout()
plt.show()

print('Pre-filtering makes the thresholded regions more coherent, but a single global threshold is still imperfect on img_ds.jpg.')


### 2.4 Automatic thresholding (Otsu)

Task:
- Apply Otsu thresholding to at least one image.
- Evaluate whether the result is relevant/acceptable.


In [ ]:
threshold_otsu_voilier, otsu_voilier = auto_foreground_otsu(img_gray)
threshold_otsu_img_ds, otsu_img_ds = auto_foreground_otsu(img_ds_gray)

plt.figure(figsize=(12, 8))
results = [
    (img_gray, 'voilier grayscale'),
    (otsu_voilier, f'voilier Otsu (T={threshold_otsu_voilier:.0f})'),
    (img_ds_gray, 'img_ds grayscale'),
    (otsu_img_ds, f'img_ds Otsu (T={threshold_otsu_img_ds:.0f})'),
]
for idx, (image, title) in enumerate(results, start=1):
    plt.subplot(2, 2, idx)
    plt.imshow(image, cmap='gray')
    plt.title(title)
    plt.axis('off')

plt.tight_layout()
plt.show()

print(f'Otsu threshold on voilier: {threshold_otsu_voilier:.0f}')
print(f'Otsu threshold on img_ds: {threshold_otsu_img_ds:.0f}')
print('Otsu is convincing on voilier because the dark birds are well separated, but it remains only moderately stable on img_ds where the classes overlap more.')


## 3. Acquisition, Visualization, and Recording (Same as TE02 Section 4)

Task:
- Reuse TE02 video pipeline in this TE context.

Checklist:
- [ ] Capture continuous webcam stream.
- [ ] Display stream live.
- [ ] Use key controls for stop and optional snapshots.
- [ ] Optionally save video stream.


In [ ]:
RUN_VIDEO_DEMOS = False

if RUN_VIDEO_DEMOS:
    run_camera_session(OUT_DIR, 'TE03 Preview', allow_snapshot=True, record_path=OUT_DIR / 'te03_webcam.avi')
else:
    print("Set RUN_VIDEO_DEMOS = True to run the TE03 webcam pipeline. Press 'q' to quit and 's' to save snapshots.")


## 4. Application to a Video Stream

Task sequence from statement:
1. Place a very bright or very dark object in front of camera background.
2. Acquire continuous stream.
3. Apply Otsu thresholding so object of interest appears white.
4. Describe, analyze, and explain observed behavior.


In [ ]:
RUN_VIDEO_DEMOS = False


def otsu_preview(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    _, binary = auto_foreground_otsu(gray)
    return binary


if RUN_VIDEO_DEMOS:
    run_camera_session(OUT_DIR, 'TE03 Otsu Stream', transform=otsu_preview)
else:
    print("Set RUN_VIDEO_DEMOS = True to run the real-time Otsu demo. Press 'q' to quit.")


## Final Self-Check

- [x] You implemented all code cells.
- [x] You answered interpretation/theory prompts.
- [x] You saved representative outputs in `../outputs/te03/`.
